<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:




```python
import pennylane as qml
import pennylane.numpy as np

def beam_splitter(r):
    t = np.sqrt(1 - r**2)
    return np.array([[r, t], [t, -r]])

dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def mz_interferometer(r):
    # Start in state |1> (photon in path 1)
    qml.BasisState(np.array([0, 1]), wires=[0, 1])

    # First beam splitter on wire 0
    BS = beam_splitter(r)
    qml.QubitUnitary(BS, wires=0)

    # Measure detector A (wire 0)
    m_a = qml.measure(0)

    # Conditioned on not detecting at A, apply second beam splitter on wire 1
    with qml.cond(~m_a):
        qml.QubitUnitary(BS, wires=1)

    # Return probabilities for final detectors C (wire 0) and D (wire 1)
    return qml.probs(wires=[0, 1])

def mz_interferometer_full(r):
    probs = mz_interferometer(r)

    # Calculate probability of detection at A (after first beam splitter)
    BS = beam_splitter(r)
    input_state = np.array([0, 1])  # |1> state
    state_after_first_bs = BS @ input_state
    p_A = np.abs(state_after_first_bs[0])**2  # probability in path 0

    # Get final probabilities
    p00, p01, p10, p11 = probs

    # Marginal probabilities
    p_C = p00 + p01  # detector C (wire 0 in |0>)
    p_D = p01 + p11  # detector D (wire 1 in |1>)

    # Total probability for AC
    p_AC = p_A + (1 - p_A) * p_C

    return np.array([p_AC, p_D])

def run(test_case_input: str) -> str:
    import json
    ins = json.loads(test_case_input)
    outs = mz_interferometer_full(ins).tolist()
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    import json
    import numpy as np
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, rtol=1e-4), "Not the correct probabilities"

test_cases = [
    ('0.5', '[0.8125, 0.1875]'),
    ('0.577350269', '[0.777778, 0.222222]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")
    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError as e:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


SyntaxError: invalid syntax (4261589568.py, line 1)